In [0]:
from pyspark.sql.functions import current_timestamp
import requests
from datetime import date, timedelta

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

caminho_base = "/Volumes/rocket_lab/default/rocket_lab/"
arquivos = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

for arquivo, tabela in arquivos.items():
    # multiLine, quote e escape porque sinopses e taglines possuem virgulas, aspas e quebras de linha internas
    df = spark.read.csv(
        f"{caminho_base}{arquivo}",
        header=True, inferSchema=True,
        multiLine=True, quote='"', escape='"'
    )
    df = df.withColumn("ingestion_datetime", current_timestamp())
    # requisito do enunciado: gravar camada Bronze em modo append
    df.write.format("delta").mode("append").saveAsTable(f"bronze.{tabela}")

dbutils.widgets.text("data_inicio", (date.today() - timedelta(days=7)).strftime("%m-%d-%Y"))
dbutils.widgets.text("data_fim", date.today().strftime("%m-%d-%Y"))
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

resposta = requests.get(url, timeout=30)
resposta.raise_for_status()
cotacoes = resposta.json()["value"]

# fins de semana e feriados fazem a PTAX devolver lista vazia, e sem esse guard o createDataFrame quebra
if not cotacoes:
    raise ValueError(f"PTAX retornou vazio para {data_inicio}..{data_fim}")

# mergeSchema defensivo para o caso da API passar a trazer alguma coluna nova sem quebrar o append
spark.createDataFrame(cotacoes) \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.tb_cotacao_dolar")